In [23]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [25]:
from dataclasses import dataclass
from typing import TypedDict

@dataclass
class Context:
    user_id: str
    app_name: str

class UserInfo(TypedDict):
    personal_info: str
    preference: str

In [26]:
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_user_info(runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    현재 사용자의 정보를 장기 메모리에서 조회합니다.
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    memories = runtime.store.search((user_id, app))
    if not memories:
        return "기록된 정보 없음"

    results = []
    for item in memories:
        data = item.value
        if "personal_info" in data:
            results.append(f"- 개인정보: {data['personal_info']}")
        if "preference" in data:
            results.append(f"- 선호도: {data['preference']}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [27]:
import uuid

@tool
def save_user_info(user_info: UserInfo, runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    사용자의 새로운 정보를 장기 메모리에 저장하거나 업데이트합니다.
    대화 중 새로운 사실이 발견되면 LLM이 스스로 판단하여 이 도구를 호출해 기억합니다.
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    memory_key = str(uuid.uuid4())
    store.put((user_id, app), memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

In [28]:
system_message = "당신은 사용자의 정보를 기억하는 비서입니다. 사용자가 자신의 정보를 말하면 반드시 'get_user_info' 도구를 사용하여 저장하세요."

In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    store=store,
    context_schema=Context,
    tools=[get_user_info, save_user_info],
    system_prompt=system_message
)

In [30]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 아이스 아메리카노를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='나는 아이스 아메리카노를 좋아해', additional_kwargs={}, response_metadata={}, id='2362e7d9-9d6d-4b1d-877d-a6e2e7e8ca78'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"personal_info": "\\uc0ac\\uc6a9\\uc790\\ub294 \\uc544\\uc774\\uc2a4 \\uc544\\uba54\\ub9ac\\uce74\\ub178\\ub97c \\uc88b\\uc544\\ud569\\ub2c8\\ub2e4.", "preference": "\\uc544\\uc774\\uc2a4 \\uc544\\uba54\\ub9ac\\uce74\\ub178"}}'}, '__gemini_function_call_thought_signatures__': {'Jzw7GqXq': 'EjQKMgEMOdbHE166zFuI3u98KnsOfffU/VnwtN2kAZCyTtSaddDN+zlXXXF/g7aoQgzpDeU3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9b91-ebb1-7d43-a798-2b9795759f49-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_info': {'personal_info': '사용자는 아이스 아메리카노를 좋아합니다.', 'preference': '아이스 아메리카노'}}, 'id': 'Jzw7GqXq', 'type': 'tool_call'}], inva

In [31]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='b86344bc-7f20-49c8-82d1-3d0aae60e9f8'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'pqUuU0tU': 'EjQKMgEMOdbHkEC0i3U/WWMd1aZz6JwrlHt7WDYS1TO+yfld5HZ/h8K+yMDeUCjqhz1NL2A1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9b91-f183-7d12-88fb-c449f5960786-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'pqUuU0tU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 12, 'total_tokens': 193, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='- 개인정보: 사용자는 아이스 아메리카노를 좋아합니다.\n- 선호도: 아이스 아메리카노', name='get_user_info', id='eae8ce72-655c-4d6a-a051-095df82ed4c9', tool_call_id='pqUuU0tU'),
  AIMessage(c

In [32]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_002", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_002...me='personal_assistant'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\wsh\langchain-workspace\ch11\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_002...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='5e1f255c-7586-4262-b77e-ae9a8a6ae0e1'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'WfApYDkQ': 'EjQKMgEMOdbHgFlkmEEjD1PSHQejPFfa/zK1fqW488vFTsrE1XRAYZse/55npwrL2XvIJbiE'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e9b93-2586-7221-abaf-7ab6f7187ace-0', tool_calls=[{'name': 'get_user_info', 'args': {}, 'id': 'WfApYDkQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 12, 'total_tokens': 193, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='기록된 정보 없음', name='get_user_info', id='baac46bb-cb10-4131-b6ba-13d1eed53fb6', tool_call_id='WfApYDkQ'),
  AIMessage(content=[{'type': 'text', 'text': '아직 당신